# 1 · EJERCICIO
- Extiende ModelConfig con una fábrica nueva y una validación nueva, sin romper las que ya existen.

1. Agrega un classmethod from_nombre(cls, nombre: str) que cree la config con  learning_rate=0.01 y epochs según el nombre: 3 si el nombre contiene 'test', 25 en caso contrario.
2. Agrega un staticmethod is_valid_epochs(e: int) -> bool que acepte solo enteros entre 1 y 1000.
3. Úsalo en __init__ igual que is_valid_lr: si los epochs no son válidos, lanza ValueError.
4. Demuestra los tres caminos: from_nombre('test-resnet'), from_nombre('resnet-final') y un intento con epochs=5000 que debe fallar.
> Pista: from_nombre decide valores y delega: sigue devolviendo cls(...). No dupliques la validación dentro de la fábrica; __init__ ya la hace.

In [ ]:
class ModelConfig:
    SUPPORTED_FRAMEWORKS: list[str] = ["pytorch", "scikit-learn", "xgboost"]

    def __init__(self, model_name: str, learning_rate: float, epochs: int = 10):
        if not ModelConfig.is_valid_lr(learning_rate):
            raise ValueError(f"learning rate fuera de rango: {learning_rate}")
        self.model_name = model_name
        self.learning_rate = learning_rate
        self.epochs = epochs
        self.is_trained = False

    @classmethod
    def default_quick_test(cls) -> Self:
        """Fábrica: configuración lista para pruebas rápidas."""
        return cls(model_name="baseline-test", learning_rate=0.001, epochs=3)

    @classmethod
    def from_dict(cls, data: dict) -> Self:
        """Fábrica: desde un dict (JSON/YAML), con defaults si faltan claves."""
        return cls(
            model_name=data.get("model_name", "custom-model"),
            learning_rate=data.get("learning_rate", 0.01),
            epochs=data.get("epochs", 5),
        )

    @classmethod 
    def from_nombre(cls, nombre:str):
        return cls(
            learning_rate =
            epochs =
            

        )

    

    @staticmethod
    def is_valid_lr(lr: float) -> bool:
        """Utilidad: no necesita ni la clase ni la instancia."""
        return 0.0 < lr <= 1.0


rapida = ModelConfig.default_quick_test()
desde_json = ModelConfig.from_dict({"model_name": "resnet", "epochs": 20})
# desde_json.learning_rate == 0.01  ← default, porque la clave no venía

In [ ]:
from typing import Self
class ModelConfig:
    SUPPORTED_FRAMEWORKS: list[str] = ["pytorch", "scikit-learn", "xgboost"]
    def __init__(self, model_name: str, learning_rate: float, epochs: int = 10):
        # La validación vive SOLO aquí: todo camino de creación pasa por __init__
        if not self.is_valid_lr(learning_rate):
            raise ValueError(f"learning_rate inválido: {learning_rate}")
        if not self.is_valid_epochs(epochs):
            raise ValueError(f"epochs inválido: {epochs} (debe ser entero entre 1 y 1000)")
        self.model_name: str = model_name
        self.learning_rate: float = learning_rate
        self.epochs: int = epochs
        self.is_trained: bool = False
    def summary(self) -> None:
        estado = "Entrenado" if self.is_trained else "Sin entrenar"
        print(f"[CONFIG] Modelo: {self.model_name:<14} | LR: {self.learning_rate:<6} | Épocas: {self.epochs:<4} | Estado: {estado}")
    # 1. Fábrica: decide valores y DELEGA en cls(...)
    @classmethod
    def from_nombre(cls, nombre: str) -> Self:
        epochs = 3 if "test" in nombre else 25
        return cls(model_name=nombre, learning_rate=0.01, epochs=epochs)
    @staticmethod
    def is_valid_lr(lr: float) -> bool:
        return 0.0 < lr <= 1.0
    # 2. Validación: no usa self ni cls → staticmethod
    @staticmethod
    def is_valid_epochs(e: int) -> bool:
        return isinstance(e, int) and not isinstance(e, bool) and 1 <= e <= 1000
# 4. Los tres caminos
ModelConfig.from_nombre("test-resnet").summary()    # epochs = 3
ModelConfig.from_nombre("resnet-final").summary()   # epochs = 25
try:
    ModelConfig("resnet", 0.01, epochs=5000)        # debe fallar
except ValueError as err:
    print(f"ValueError: {err}")